<a href="https://colab.research.google.com/github/mahaapoorani/Machine-Learning-Projects/blob/main/Random_forest_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
df= pd.read_csv("/content/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
df.isnull().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [19]:
df["total_bedrooms"]=df["total_bedrooms"].fillna(df["total_bedrooms"].mean())

In [4]:
df.isnull().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [5]:
X= df.drop("median_house_value",axis=1)
y=df["median_house_value"]
X.shape
y.shape

(20640,)

In [6]:
X=pd.get_dummies(X, columns=["ocean_proximity"])

In [7]:
X.shape

(20640, 13)

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size= 0.2, random_state= 42)

In [22]:
from sklearn.ensemble import RandomForestRegressor
model= RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [23]:
from sklearn.model_selection import RandomizedSearchCV
param_distributions = {

    "n_estimators":[50,100,200,300],

    "max_depth":[10,12,15,20],

    "min_samples_split":[2,5,10]

}
random_search = RandomizedSearchCV(

    estimator=model,

    param_distributions=param_distributions,n_iter=10,

    cv=5,

    scoring="r2",
    random_state=42,

    n_jobs=-1

)

In [24]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 12, 15, 20],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200, 300]},
                   random_state=42, scoring='r2')

In [25]:
print(random_search.best_params_)

{'n_estimators': 300, 'min_samples_split': 5, 'max_depth': 20}


In [26]:
print(random_search.best_score_)

0.8187326490961999


In [27]:
best_model=random_search.best_estimator_

In [28]:
y_pred=best_model.predict(X_test)

In [29]:
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Train Score:", best_model.score(X_train, y_train))
print("Test Score:", best_model.score(X_test, y_test))
print("MAE:", mae)
print("R² Score:", r2)

# Actual vs Predicted

print("\nActual vs Predicted:\n")

for actual, predicted in zip(y_test[:5], y_pred[:5]):
    print(f"ACTUAL: {actual:.2f}")
    print(f"PREDICTED: {predicted:.2f}")
    print("-" * 30)

Train Score: 0.9626214178970727
Test Score: 0.8185907855922313
MAE: 31548.656754205073
R² Score: 0.8185907855922313

Actual vs Predicted:

ACTUAL: 47700.00
PREDICTED: 51592.83
------------------------------
ACTUAL: 45800.00
PREDICTED: 68930.34
------------------------------
ACTUAL: 500001.00
PREDICTED: 473008.97
------------------------------
ACTUAL: 218600.00
PREDICTED: 250187.51
------------------------------
ACTUAL: 278000.00
PREDICTED: 262528.97
------------------------------


In [29]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
kf= KFold (n_splits=5, shuffle= True, random_state= 42)
scores= cross_val_score(model,X,y,cv=kf, scoring= "r2")
print("\n Cross validation Scores")
print(scores)
print("\nAverage R2:", scores.mean())
print("Standard deviation", scores.std())


 Cross validation Scores
[0.81780885 0.82845883 0.81985604 0.84086643 0.8137624 ]

Average R2: 0.8241505118682666
Standard deviation 0.009639038530892922


In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

                       Feature  Importance
7                median_income    0.490384
9       ocean_proximity_INLAND    0.141366
0                    longitude    0.105698
1                     latitude    0.101170
2           housing_median_age    0.052319
5                   population    0.032625
3                  total_rooms    0.023652
4               total_bedrooms    0.023641
6                   households    0.018241
12  ocean_proximity_NEAR OCEAN    0.006415
8    ocean_proximity_<1H OCEAN    0.003394
11    ocean_proximity_NEAR BAY    0.000714
10      ocean_proximity_ISLAND    0.000381
